# Smoke test: BGE reranker fine-tuning

This is a plumbing test only: one query, one positive passage, five negative passages, and one optimizer step. It verifies model download, tokenization, CUDA forward/backward, FlagEmbedding training, and checkpoint saving without encoding the legal corpus. It is not an experiment and produces no reportable metric.

In [ ]:
from kaggle_secrets import UserSecretsClient
from pathlib import Path
import os

token = UserSecretsClient().get_secret('HF_TOKEN')
if not token:
    raise RuntimeError('Kaggle Secret HF_TOKEN is missing')
os.environ['HF_TOKEN'] = token
WORK = Path('/kaggle/working/reranker_smoke_test')
WORK.mkdir(parents=True, exist_ok=True)

In [ ]:
!python -m pip install 'FlagEmbedding==1.4.2'

In [ ]:
import json
import torch

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable')
print('GPU:', torch.cuda.get_device_name(0))
train_path = WORK / 'one_query.jsonl'
record = {
    'query': 'Phạm nhân không biết chữ có được học văn hóa để xóa mù chữ không?',
    'pos': ['Phạm nhân chưa biết chữ được tạo điều kiện học văn hóa để xóa mù chữ.'],
    'neg': [
        'Quy định về mở và sử dụng tài khoản vốn bằng ngoại tệ.',
        'Điều kiện cấp giấy phép xây dựng đối với nhà ở riêng lẻ.',
        'Thủ tục đăng ký thay đổi nội dung đăng ký doanh nghiệp.',
        'Quy định về thuế đối với hoạt động xuất khẩu hàng hóa.',
        'Trình tự giải quyết tranh chấp hợp đồng dân sự.',
    ],
}
train_path.write_text(json.dumps(record, ensure_ascii=False) + '\n', encoding='utf-8')
print(train_path.read_text(encoding='utf-8'))

## One GPU, one optimizer step

The full notebook uses two-GPU DDP. This smoke test deliberately uses one GPU because one training group cannot be distributed across two workers safely.

In [ ]:
MODEL_OUT = WORK / 'model'
!CUDA_VISIBLE_DEVICES=0 python -m FlagEmbedding.finetune.reranker.encoder_only.base \
  --model_name_or_path BAAI/bge-reranker-v2-m3 \
  --train_data {train_path} \
  --cache_path {WORK / 'cache'} \
  --train_group_size 6 \
  --query_max_len 64 \
  --passage_max_len 448 \
  --pad_to_multiple_of 8 \
  --knowledge_distillation False \
  --output_dir {MODEL_OUT} \
  --learning_rate 2e-5 \
  --fp16 \
  --max_steps 1 \
  --per_device_train_batch_size 1 \
  --gradient_accumulation_steps 1 \
  --dataloader_drop_last False \
  --gradient_checkpointing \
  --weight_decay 0.01 \
  --warmup_ratio 0.0 \
  --logging_steps 1 \
  --save_strategy no \
  --report_to none

In [ ]:
required = [MODEL_OUT / 'config.json']
weight_candidates = [MODEL_OUT / 'model.safetensors', MODEL_OUT / 'pytorch_model.bin']
if not all(path.is_file() for path in required) or not any(path.is_file() for path in weight_candidates):
    raise RuntimeError(f'Smoke training finished but model files are incomplete: {list(MODEL_OUT.iterdir())}')
print('SMOKE TEST PASSED')
print('Saved model:', MODEL_OUT)
print('Files:', sorted(path.name for path in MODEL_OUT.iterdir()))